In [ ]:
import sys
import os

sys.path.insert(0, os.path.join(os.getcwd(), ".."))


# Original analyzer for data loading
from src.analysis.summary import EEGSummarizedAnalyzer
from src.definitions.fields import (
    ExperimentNames,
    CoordinateSystems,
    ConditionVariants,
    MusicTypeVariants,
    ExclusionCategories,
    SingleDataMetadata,
)
from src.definitions.fields import (
    ConditionVariants,
    MusicTypeVariants,
    ExclusionCategories,
)
from src.definitions.constants import ProjectPaths
# Modular analysis pipeline
from src.analysis.isc import (
    compute_mean_variance,
    compute_sliding_window_mean_variance,
)
from src.visualization.isc_plots import (
    plot_mean_variance_distribution,
    plot_sliding_window_mean_variance,
    plot_band_mean_variance_distributions,
    plot_band_sliding_window_mean_variance,
    print_data_overview,
)
from scripts.analysis_common import (
    load_analyzers,
    analyzers_to_datasets,
    run_isc_workflow,
    run_mean_variance_workflow,
)

%matplotlib inline

In [ ]:
# Condition and music types
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# Sliding-window parameters (shared)
WINDOW_SEC = 0.04
STEP_SEC = 0.04

# Set to True to load raw files, resample, stack and save before analysis
process_and_save_data = False

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES, 
    CONDITION, 
    EXCLUSION_CATEGORIES, 
    process_and_save_data,
    normalize_data=True
)
datasets = analyzers_to_datasets(analyzers)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Dataset selection ──────────────────────────────────────────────────────
# Change LABEL to switch between conditions; data shape: (n_subjects, n_channels, n_times)
LABEL = MusicTypeVariants.CLASSICAL.value
# LABEL = MusicTypeVariants.PSYTRANCE.value

ad = datasets[LABEL]
data = ad.data        # (n_subjects, n_channels, n_times)
sfreq = ad.sfreq

n_subjects, n_channels, n_times = data.shape
time = np.arange(n_times) / sfreq  # seconds

print(f"Dataset : {LABEL}")
print(f"Shape   : {data.shape}  (subjects × channels × time points)")
print(f"Duration: {n_times / sfreq:.1f} s  @  {sfreq} Hz")


## Section 1 — Intersubject Mean & Variance per Time Frame

At each time point we summarise **how participants agree** by computing:

- **Intersubject mean** — mean across subjects, averaged over channels → shared response
- **Intersubject variance** — variance *across subjects* at each sample, averaged over channels  
  → moment-to-moment synchrony (low = participants are in sync)
- **Per-subject traces** — each subject's channel-average: reveals individual outliers


In [ ]:
# Variance across subjects at each (channel, time) -> (n_channels, n_times)
inter_var  = data.var(axis=0)    # low = subjects agree here
inter_mean = data.mean(axis=0)   # (n_channels, n_times)

# Collapse over channels -> scalar time series
mean_t = inter_mean.mean(axis=0)   # (n_times,)
var_t  = inter_var.mean(axis=0)    # (n_times,)
std_t  = np.sqrt(var_t)

# Per-subject channel-average -> (n_subjects, n_times)
mean_over_ch = data.mean(axis=1)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# --- Panel 1: Mean signal with per-subject trajectories ---
ax = axes[0]
for s in range(n_subjects):
    ax.plot(time, mean_over_ch[s], color="steelblue", alpha=0.2, linewidth=0.7)
ax.plot(time, mean_t, color="black", linewidth=2.0, label="Group mean (across subjects & channels)")
ax.fill_between(time, mean_t - std_t, mean_t + std_t,
                color="steelblue", alpha=0.22, label="±1 SD (intersubject)")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.7)
ax.set_ylabel("Signal (z-score)")
ax.set_title(f"[{LABEL}]  Intersubject mean signal over time (electrode-averaged)")
ax.legend(frameon=False, fontsize=9)

# --- Panel 2: Mean intersubject variance over time ---
ax = axes[1]
ax.plot(time, var_t, color="darkorange", linewidth=1.8,
        label="Mean intersubject variance (across channels)")
ax.fill_between(time, 0, var_t, color="darkorange", alpha=0.18)
ax.axhline(var_t.mean(), color="gray", linestyle="--", linewidth=0.8,
           label=f"Grand mean variance ({var_t.mean():.3f})")
ax.set_ylabel("Variance")
ax.set_xlabel("Time (s)")
ax.set_title("Intersubject variance over time — low = participants are in sync")
ax.legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.show()


## Section 2 — Distribution of Intersubject Variance

Instead of a time-series view, here we look at the **distribution** of the
intersubject variance scores across all `(channel × time)` samples.

- **Histogram** — clipped at the `PLOT_PCT`th percentile to suppress extreme outliers; the number of clipped samples is printed and marked on the plot.
- **Per-channel between-subject comparison** — boxplot + individual subject dots per electrode (sorted by median variance); each dot is one subject's signal variance over time for that channel, making between-subject differences immediately visible.


In [ ]:
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)

# ── Outlier threshold ─────────────────────────────────────────────────────
PLOT_PCT  = 99   # clip plots at this percentile to suppress outliers

all_var   = inter_var.ravel()
p_clip    = np.percentile(all_var, PLOT_PCT)
n_outlier = int((all_var > p_clip).sum())
frac_out  = 100.0 * n_outlier / all_var.size
total_samples = all_var.size

print(f"Variance stats — mean: {all_var.mean():.4f}  median: {np.median(all_var):.4f}"
      f"  std: {all_var.std():.4f}  max: {all_var.max():.4f}")
print(f"Outliers (>{PLOT_PCT}th pct = {p_clip:.4f}): {n_outlier:,} samples "
      f"({frac_out:.2f}% of all channel×time points) — excluded from plot below\n")

# ── Per-subject per-channel variance (used later for the heatmap) ─────────
per_subj_ch_var = data.var(axis=2)   # (n_subjects, n_channels)

# ── Histogram (percentage y-axis, clipped at PLOT_PCT) ───────────────────
clipped_vals = all_var[all_var <= p_clip]

fig, ax = plt.subplots(figsize=(9, 5))

counts, bin_edges = np.histogram(clipped_vals, bins=80)
pct_vals = counts / total_samples * 100          # convert to % of ALL samples
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
bar_width = bin_edges[1] - bin_edges[0]

ax.bar(bin_centers, pct_vals, width=bar_width * 0.95,
       color="darkorange", alpha=0.85, edgecolor="none")
ax.axvline(np.median(all_var), color=".2", linestyle="--", linewidth=1.2,
           label=f"Median = {np.median(all_var):.3f}")
ax.axvline(p_clip, color="crimson", linestyle=":", linewidth=1.4,
           label=f"{PLOT_PCT}th pct = {p_clip:.3f}")
ax.set_xlabel("Intersubject variance")
ax.set_ylabel("% of all channel × time samples")
ax.set_title(f"[{LABEL}]  Variance distribution (clipped at {PLOT_PCT}th pct)\n"
             f"⚠ {n_outlier:,} outlier samples ({frac_out:.2f}%) not shown")
ax.legend(frameon=False, fontsize=9)

sns.despine(fig=fig)
fig.tight_layout()
plt.show()


## Section 4 — Windowed Mean & Variance

The recording is divided into short, EEG-relevant non-overlapping time windows. For each window we compute:

- **Mean signal** — average of the group-mean signal across that window (±1 SD across subjects)
- **Mean variance** — average intersubject variance across that window

Two figures are produced:
1. **Bar charts** — one bar per window, green = synchrony candidate (mean variance below `SYNC_PERCENTILE`th percentile)
2. **Overlay line plots** — the continuous signal/variance time series with the step-wise windowed mean superimposed, making it easy to see how the windowed summaries track the raw signal


In [ ]:
import seaborn as sns
import pandas as pd
from matplotlib.patches import Patch

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)

# ── Parameters ────────────────────────────────────────────────────────────
WINDOW_DURATION_SEC = 2.0   # EEG-relevant window (seconds); tweak here

# ── Build per-window statistics ───────────────────────────────────────────
win_samples = int(WINDOW_DURATION_SEC * sfreq)
n_windows   = n_times // win_samples

records = []
for w in range(n_windows):
    sl = slice(w * win_samples, (w + 1) * win_samples)
    subj_mean = mean_over_ch[:, sl].mean(axis=1)   # (n_subjects,)
    records.append({
        "window":        w + 1,
        "center":        time[w * win_samples + win_samples // 2],
        "t_start":       time[w * win_samples],
        "t_end":         time[min((w + 1) * win_samples - 1, n_times - 1)],
        "mean_signal":   subj_mean.mean(),
        "var_signal":    subj_mean.var(),
        "mean_variance": var_t[sl].mean(),
    })

df_wins = pd.DataFrame(records)
df_wins["sync_candidate"] = df_wins["mean_variance"] < sync_threshold

sync_label = f"Sync candidate (var < {SYNC_PERCENTILE}th pct)"
norm_label = "Normal window"
df_wins["category"] = df_wins["sync_candidate"].map({True: sync_label, False: norm_label})

win_centers  = df_wins["center"].values
win_mean_sig = df_wins["mean_signal"].values
win_var_sig  = df_wins["var_signal"].values
win_mean_var = df_wins["mean_variance"].values
is_sync_win  = df_wins["sync_candidate"].values

# Step-function arrays for overlay (padded to n_times in case of remainder samples)
_pad = n_times - n_windows * win_samples
win_mean_sig_step = np.pad(np.repeat(win_mean_sig, win_samples), (0, _pad), mode='edge')
win_mean_var_step = np.pad(np.repeat(win_mean_var, win_samples), (0, _pad), mode='edge')

# ── Colour palettes ───────────────────────────────────────────────────────
muted = sns.color_palette("muted")
C_BLUE   = muted[0]   # signal normal
C_ORANGE = muted[1]   # variance normal
C_GREEN  = "#5cb85c"  # sync candidate
C_RED    = muted[2]   # windowed variance step line

sig_palette = {sync_label: C_GREEN, norm_label: C_BLUE}
var_palette = {sync_label: C_GREEN, norm_label: C_ORANGE}

bar_colors_sig = [C_GREEN if s else C_BLUE   for s in is_sync_win]
bar_colors_var = [C_GREEN if s else C_ORANGE for s in is_sync_win]

# ════════════════════════════════════════════════════════════════════════════
# Figure 1 — Bar charts
# ════════════════════════════════════════════════════════════════════════════
bar_w = WINDOW_DURATION_SEC * 0.82
fig1, axes = plt.subplots(2, 1, figsize=(max(14, n_windows * 0.18), 7), sharex=True)

# --- Panel 1: mean signal per window ---
ax = axes[0]
ax.bar(win_centers, win_mean_sig, width=bar_w,
       color=bar_colors_sig, alpha=0.85, edgecolor="none")
ax.errorbar(win_centers, win_mean_sig, yerr=win_var_sig,
            fmt="none", color=".3", capsize=2, linewidth=0.8,
            label="±variance across subjects")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, label="Zero baseline")
ax.axhline(win_mean_sig.mean(), color=".2", linestyle=":", linewidth=1.1,
           label=f"Grand mean ({win_mean_sig.mean():.3f})")
ax.set_ylabel("Mean signal (z-score)")
ax.set_title(f"[{LABEL}]  Per-window mean signal  (window = {WINDOW_DURATION_SEC:.1f} s)")
bar_patches = [Patch(color=c, alpha=0.85, label=l) for l, c in sig_palette.items()]
extra_h, extra_l = ax.get_legend_handles_labels()
ax.legend(handles=bar_patches + extra_h,
          labels=[p.get_label() for p in bar_patches] + extra_l,
          frameon=False, fontsize=9)

# --- Panel 2: mean variance per window ---
ax = axes[1]
ax.bar(win_centers, win_mean_var, width=bar_w,
       color=bar_colors_var, alpha=0.85, edgecolor="none")
ax.axhline(sync_threshold, color=C_GREEN, linestyle="--", linewidth=1.1,
           label=f"Sync threshold — {SYNC_PERCENTILE}th pct ({sync_threshold:.3f})")
ax.axhline(win_mean_var.mean(), color=".2", linestyle=":", linewidth=1.1,
           label=f"Grand mean variance ({win_mean_var.mean():.3f})")
ax.set_ylabel("Mean intersubject variance")
ax.set_xlabel("Time (s)")
ax.set_title("Per-window mean intersubject variance  |  green = sync candidates")
var_patches = [Patch(color=c, alpha=0.85, label=l) for l, c in var_palette.items()]
extra_h2, extra_l2 = ax.get_legend_handles_labels()
ax.legend(handles=var_patches + extra_h2,
          labels=[p.get_label() for p in var_patches] + extra_l2,
          frameon=False, fontsize=9)

sns.despine(fig=fig1, left=False, bottom=False)
fig1.tight_layout()
plt.show()

# ════════════════════════════════════════════════════════════════════════════
# Figure 2 — Continuous overlay with seaborn lineplots
# ════════════════════════════════════════════════════════════════════════════
# Downsample for performance (seaborn lineplot on tens-of-thousands of pts is slow)
ds = max(1, n_times // 8000)
t_ds = time[::ds]

df_sig = pd.DataFrame({
    "time":     t_ds,
    "mean":     mean_t[::ds],
    "lower":    (mean_t - std_t)[::ds],
    "upper":    (mean_t + std_t)[::ds],
    "windowed": win_mean_sig_step[::ds],
})
df_var = pd.DataFrame({
    "time":     t_ds,
    "variance": var_t[::ds],
    "windowed": win_mean_var_step[::ds],
})

fig2 = plt.figure(figsize=(15, 11))
# 2-column GridSpec: column 0 = all plots, column 1 = colorbar only.
# This keeps all three main axes at identical width regardless of the colorbar.
gs2  = gridspec.GridSpec(
    3, 2,
    height_ratios=[2, 2, 3],
    width_ratios=[30, 1],
    hspace=0.20, wspace=0.05,
    figure=fig2,
)
ax_s   = fig2.add_subplot(gs2[0, 0])
ax_v   = fig2.add_subplot(gs2[1, 0], sharex=ax_s)
ax_hm2 = fig2.add_subplot(gs2[2, 0], sharex=ax_s)
cax    = fig2.add_subplot(gs2[2, 1])   # dedicated colorbar axes

# --- Panel 1: signal overlay ---
ax = ax_s
ax.fill_between(df_sig["time"], df_sig["lower"], df_sig["upper"],
                color=C_BLUE, alpha=0.15, label="±1 SD (intersubject)")
sns.lineplot(data=df_sig, x="time", y="mean", ax=ax, errorbar=None,
             color=".2", linewidth=1.1, alpha=0.6, label="Continuous group mean")
ax.step(df_sig["time"], df_sig["windowed"], where="post",
        color=C_BLUE, linewidth=2.0, label=f"Windowed mean ({WINDOW_DURATION_SEC:.1f} s)")
first_s = True
for w in np.where(is_sync_win)[0]:
    t_s = time[w * win_samples]
    t_e = time[min((w + 1) * win_samples - 1, n_times - 1)]
    ax.axvspan(t_s, t_e, color=C_GREEN, alpha=0.22,
               label="Sync candidate window" if first_s else "_nolegend_")
    first_s = False
ax.axhline(0, color="gray", linestyle="--", linewidth=0.7, label="Zero baseline")
ax.set_ylabel("Signal (z-score)")
ax.set_title(f"[{LABEL}]  Continuous vs. windowed mean signal / variance + per-electrode heatmap  (window = {WINDOW_DURATION_SEC:.1f} s)")
ax.legend(frameon=False, fontsize=9)
plt.setp(ax.get_xticklabels(), visible=False)

# --- Panel 2: variance overlay ---
ax = ax_v
sns.lineplot(data=df_var, x="time", y="variance", ax=ax, errorbar=None,
             color=C_ORANGE, linewidth=1.0, alpha=0.55, label="Continuous intersubject variance")
ax.fill_between(df_var["time"], 0, df_var["variance"], color=C_ORANGE, alpha=0.10)
ax.step(df_var["time"], df_var["windowed"], where="post",
        color=C_RED, linewidth=2.0, label=f"Windowed mean variance ({WINDOW_DURATION_SEC:.1f} s)")
ax.axhline(sync_threshold, color=C_GREEN, linestyle="--", linewidth=1.1,
           label=f"Sync threshold — {SYNC_PERCENTILE}th pct ({sync_threshold:.3f})")
first_s = True
for w in np.where(is_sync_win)[0]:
    t_s = time[w * win_samples]
    t_e = time[min((w + 1) * win_samples - 1, n_times - 1)]
    ax.axvspan(t_s, t_e, color=C_GREEN, alpha=0.22,
               label="Sync candidate window" if first_s else "_nolegend_")
    first_s = False
ax.set_ylabel("Intersubject variance")
ax.legend(frameon=False, fontsize=9)
plt.setp(ax.get_xticklabels(), visible=False)

# --- Panel 3: per-electrode variance heatmap (electrodes in natural order) ---
# pcolormesh uses true data coordinates → aligns perfectly with the shared x-axis.
# Downsample columns for rendering performance.
ds_hm     = max(1, n_times // 2000)
_time_hm  = time[::ds_hm]
_var_hm   = inter_var[:, ::ds_hm]
_dt       = (_time_hm[1] - _time_hm[0]) if len(_time_hm) > 1 else 1.0 / sfreq
_t_edges  = np.r_[_time_hm - _dt / 2, _time_hm[-1] + _dt / 2]
_ch_edges = np.arange(n_channels + 1)

pcm = ax_hm2.pcolormesh(
    _t_edges, _ch_edges, _var_hm,
    cmap="YlOrRd",
    vmin=0,
    vmax=np.percentile(inter_var, 98),
    rasterized=True,
    shading="flat",
)
ax_hm2.set_ylim(n_channels, 0)   # electrode 0 at top
# Overlay sync-candidate spans on the heatmap too
fig2.colorbar(pcm, cax=cax, label="Intersubject variance")
first_s = True
for w in np.where(is_sync_win)[0]:
    t_s = time[w * win_samples]
    t_e = time[min((w + 1) * win_samples - 1, n_times - 1)]
    ax_hm2.axvspan(t_s, t_e, color=C_GREEN, alpha=0.18,
                   label="Sync candidate window" if first_s else "_nolegend_")
    first_s = False
ax_hm2.set_xlabel("Time (s)")
ax_hm2.set_ylabel("Electrode (natural order)")

sns.despine(fig=fig2, left=False, bottom=False)
fig2.tight_layout()
plt.show()

# ── Print synchrony-candidate windows ────────────────────────────────────
sync_df = df_wins[df_wins["sync_candidate"]][["window", "t_start", "t_end", "mean_variance", "mean_signal"]]
print(f"\nWindow size: {WINDOW_DURATION_SEC:.1f} s  |  Total windows: {n_windows}")
print(f"Synchrony threshold: {sync_threshold:.4f}  ({SYNC_PERCENTILE}th percentile of var_t)")
print(f"\nSynchrony-candidate windows ({len(sync_df)}):\n")
print(sync_df.to_string(index=False, float_format="%.4f"))